# 📊 EDA — Climate-Aware Crop Yield Forecasting
Exploratory Data Analysis on sample satellite, weather, and soil data.
This notebook validates our feature engineering assumptions before model training.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style='darkgrid')
plt.rcParams['figure.figsize'] = (12, 5)
print('Libraries loaded ✅')

## 1. Load Sample Data

In [ ]:
sat_df     = pd.read_csv('../sample_satellite.csv')
weather_df = pd.read_csv('../sample_weather.csv')
soil_df    = pd.read_csv('../sample_soil.csv')

print('Satellite shape:', sat_df.shape)
print('Weather shape:  ', weather_df.shape)
print('Soil shape:     ', soil_df.shape)
sat_df.head()

## 2. Satellite — NDVI Over Growing Season

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# NDVI Time Series
axes[0].plot(sat_df['NDVI'], marker='o', color='green', linewidth=2)
axes[0].axhline(0.5, color='gray', linestyle='--', label='Healthy Vegetation Threshold')
axes[0].set_title('NDVI Over Growing Season')
axes[0].set_xlabel('Time Step (Biweekly)')
axes[0].set_ylabel('NDVI')
axes[0].legend()

# Band Correlation Heatmap
corr = sat_df.corr()
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', ax=axes[1])
axes[1].set_title('Satellite Band Correlation')

plt.tight_layout()
plt.show()

## 3. Weather — Seasonal Climate Patterns

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

axes[0, 0].plot(weather_df['temp_min_C'], label='Min Temp', color='blue')
axes[0, 0].plot(weather_df['temp_max_C'], label='Max Temp', color='red')
axes[0, 0].set_title('Temperature Range (°C)')
axes[0, 0].legend()

axes[0, 1].bar(range(len(weather_df)), weather_df['rainfall_mm'], color='steelblue')
axes[0, 1].set_title('Monthly Rainfall (mm)')

axes[1, 0].plot(weather_df['SPI_3month'], color='darkorange', marker='s')
axes[1, 0].axhline(0, color='black', linewidth=0.8)
axes[1, 0].axhline(-1, color='red', linestyle='--', label='Moderate Drought')
axes[1, 0].set_title('Standardized Precipitation Index (SPI-3)')
axes[1, 0].legend()

sns.boxplot(data=weather_df[['temp_min_C', 'temp_max_C', 'humidity_pct']], ax=axes[1, 1], palette='Set2')
axes[1, 1].set_title('Climate Variable Distributions')

plt.tight_layout()
plt.show()

## 4. Soil — Static Property Profile

In [ ]:
print('📋 Soil Properties for study area:')
print(soil_df.T.rename(columns={0: 'Value'}).to_string())

# Soil texture classification
from src.features.soil_features import SoilFeatureExtractor
extractor = SoilFeatureExtractor({})
texture = extractor.categorize_soil_texture(
    clay=soil_df['clay_content_pct'].values[0], silt=20.0, sand=52.0
)
print(f'\n🌱 Estimated USDA Soil Texture Class: {texture}')

## 5. Key EDA Findings

| Finding | Implication |
|---|---|
| NDVI peaks at step 6-8 (mid-season) | Aligns with rice tillering phase in Eastern India |
| High B8 (NIR) correlates with NDVI | NIR is primary signal — confirms band selection |
| SPI-3 dips below -1 in some months | Drought stress events present — SPI is a critical feature |
| Soil pH = 6.5, SOC = 1.8% | Slightly acidic, moderate fertility — typical for Bengal delta |
| Temp range: 16-37°C | Within GDD base/cap range (10-35°C) — GDD accumulation is meaningful |